# Task performance

Reproduces the task-performance results: **Fig. 1e**, **Fig. 1f**, **Fig. S1** and **Fig. S2**.

Compares three RNN classes trained on the delayed match-to-sample with distractors (DMS-D) task:

| Class | Projection constraints | Geometry prior |
|---|---|---|
| Vanilla RNN | – | – |
| Masked RNN | ✓ | – |
| bioRNN | ✓ | ✓ (Euclidean) |

**Requires:** trained models for rows 0–2 of the params CSV, in the `model_dir` set in `paths.yaml`.
**Does not require** empirical fMRI — this notebook runs without anything in `data_private/`.

The analysis itself lives in [`src/performance.py`](../src/performance.py); this notebook loads, computes and plots.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import src.performance as perf
import src.utils as utils
from src.config import ensure_dir, get_paths

MODEL_PARAMS = 'model_params_202606d'   # params CSV defining the training sweep
ROWS = (0, 1, 2)                        # Vanilla, Masked, bioRNN
SAVE_FIGURES = True

# Figures are written next to the models; redirect by setting figure_dir in paths.yaml.
paths = get_paths(MODEL_PARAMS)         # no fMRI needed, so require='all' is not used
figdir = ensure_dir(paths.figure_dir)

utils.set_font_size(11)
plt.rcParams['svg.fonttype'] = 'none'   # keep text editable in the saved SVGs
sns.set_style('white')
COLORS = utils.get_my_colors(cat_trio=True, as_list=True)


def save(fig, name):
    if SAVE_FIGURES:
        fig.savefig(os.path.join(figdir, name), dpi=300,
                    bbox_inches='tight', pad_inches=0.01)


print(f'models : {paths.model_dir}')
print(f'figures: {figdir}')

In [ ]:
# Load the full training trace for each class. Nothing is truncated:
# accuracy is reported at the final epoch and the fits use the whole trajectory.
data = perf.load_performance(MODEL_PARAMS, rows=ROWS)
labels = [d['label'] for d in data]

for d in data:
    print(f"{d['label']:<12} runs={d['n_runs']:3d}  "
          f"epochs={d['epochs'][0]}-{d['epochs'][-1]} (every {d['log_freq']})")

## Fig. 1e — task accuracy across training

Mean ± 95% CI over runs. Dashed verticals mark each class's training criterion: the 95th
percentile of its per-run convergence epochs (fitted below).

In [ ]:
# Logistic fits per run (full trajectory) -> convergence epochs -> class criterion.
fits = [perf.fit_runs(d['epochs'], d['accuracy']) for d in data]
criteria = [perf.criterion_epoch(f['t_conv']) for f in fits]

for lab, f, c in zip(labels, fits, criteria):
    print(f'{lab:<12} fitted={f["run_index"].size:3d}  '
          f'excluded={f["n_excluded"]:2d}  criterion={c:,.0f} epochs')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for d, color, crit in zip(data, COLORS, criteria):
    x, acc = d['epochs'], d['accuracy']
    mean = acc.mean(axis=0)
    ci = 1.96 * acc.std(axis=0) / np.sqrt(acc.shape[0])
    ax.plot(x, mean, color=color, lw=2, label=d['label'])
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.15, lw=0)
    ax.axvline(crit, color=color, ls='--', lw=1, alpha=0.8)

ax.axhline(perf.CHANCE_ACCURACY, color='0.5', ls=':', lw=1)
ax.text(x[-1], perf.CHANCE_ACCURACY + 1.5, 'chance', color='0.4',
        ha='right', va='bottom', fontsize=9)

ax.set_xlabel('Training epoch')
ax.set_ylabel('Test accuracy (%)')
ax.set_ylim(0, 105)
ax.set_yticks(np.arange(0, 101, 20))
ax.legend(frameon=False, loc='lower right')
sns.despine(fig=fig, right=True, top=True)
save(fig, 'fig1e_accuracy_across_training.svg')
plt.show()

## Fig. 1f — epochs to criterion

Per-run convergence epochs by class, compared with Wilcoxon signed-rank tests.

Runs are **paired by index**: a given run index reproduces the same initialization, trial
stream and noise draws in every class, so differences isolate the spatial constraints.

In [ ]:
comparisons = perf.compare_classes(fits, labels)

print(f'{"comparison":<28}{"n":>5}{"p":>12}{"p (Holm)":>12}{"r_rb":>8}\n' + '-' * 65)
for c in comparisons:
    print(f'{c["a"] + " vs " + c["b"]:<28}{c["n_pairs"]:>5}{c["p"]:>12.2e}'
          f'{c["p_holm"]:>12.2e}{c["rank_biserial"]:>8.2f}  '
          f'{perf.significance_stars(c["p_holm"])}')

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
rng = np.random.default_rng(0)

for i, (f, color) in enumerate(zip(fits, COLORS)):
    vals = f['t_conv']
    body = ax.violinplot(vals, positions=[i], showextrema=False)
    for b in body['bodies']:
        b.set_facecolor(color)
        b.set_alpha(0.4)
    ax.scatter(i + rng.uniform(-0.08, 0.08, vals.size), vals,
               s=10, color=color, alpha=0.6, linewidths=0.5)
    ax.hlines(np.median(vals), i - 0.22, i + 0.22, color='k', lw=1.5, zorder=3)

# Significance brackets, shortest span lowest so they nest cleanly.
idx = {lab: i for i, lab in enumerate(labels)}
top = max(f['t_conv'].max() for f in fits)
step = 0.12 * top
for level, c in enumerate(sorted(comparisons,
                                 key=lambda c: abs(idx[c['a']] - idx[c['b']]))):
    i, j = idx[c['a']], idx[c['b']]
    y = top + step * (level + 1)
    ax.plot([i, i, j, j], [y - step * 0.15, y, y, y - step * 0.15], lw=1, color='k')
    ax.text((i + j) / 2, y, perf.significance_stars(c['p_holm']),
            ha='center', va='bottom', fontsize=10)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Epochs to criterion')
ax.set_ylim(bottom=0)
sns.despine(fig=fig, right=True, top=True)
save(fig, 'fig1f_epochs_to_criterion.svg')
plt.show()

## Fig. S1 — logistic fits to the learning curves

Each class's mean accuracy curve with its logistic fit overlaid, and the distribution of
per-run fit quality.

In [ ]:
fig, axes = plt.subplots(len(data), 1, figsize=(5., 3 * len(data)), sharey=True, sharex=True)

for ax, d, f, color in zip(np.atleast_1d(axes), data, fits, COLORS):
    x, mean = d['epochs'], d['accuracy'].mean(axis=0)
    fit = perf.fit_logistic(x, mean)

    ax.plot(x, mean, color=color, lw=2, label='mean accuracy')
    if fit is not None:
        ax.plot(x, perf.logistic(x, fit['a'], fit['k'], fit['t0']),
                color='k', ls='--', lw=1.5,
                label=f"logistic fit ($R^2$ = {fit['r2']:.2f})")
    ax.set_title(f"{d['label']}\nper-run $R^2$ = "
                 f"{np.nanmean(f['r2']):.2f} ± {np.nanstd(f['r2'], ddof=1):.2f}",
                 fontsize=10)
    ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.set_xlabel('Training epoch')

np.atleast_1d(axes)[0].set_ylabel('Test accuracy (%)')
np.atleast_1d(axes)[0].set_ylim(0, 105)
sns.despine(fig=fig, right=True, top=True)
fig.tight_layout()
save(fig, 'figS1_logistic_fits.svg')
plt.show()

## Fig. S2 — task and spatial loss across training

The spatial penalty is heaviest for bioRNNs (they carry the geometry term on top of the
shared L2), yet their task loss is *lower* than the Masked RNNs' — the spatial embedding
helps rather than hinders learning.

In [ ]:
# We plot a strided subset of loss values because the generated SVG is too large otherwise.
LOSS_PLOT_POINTS = 1000

fig, axes = plt.subplots(2, 1, figsize=(5, 8), sharex=True)

for ax, key, title in zip(axes, ['loss_task', 'loss_spatial'],
                          ['Task loss (cross-entropy)', 'Spatial loss (regularization)']):
    for d, color in zip(data, COLORS):
        loss = d[key]
        if loss is None:
            continue
        step = max(1, loss.shape[1] // LOSS_PLOT_POINTS)
        x = np.arange(1, loss.shape[1] + 1)[::step]
        mean = loss.mean(axis=0)[::step]
        ci = (1.96 * loss.std(axis=0) / np.sqrt(loss.shape[0]))[::step]
        ax.plot(x, mean, color=color, lw=1.5, label=d['label'])
        ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.15, lw=0)
    ax.set_title(title, fontsize=10)
ax.set_xlabel('Training epoch')

axes[0].set_ylabel('Loss')
axes[0].legend(frameon=False, loc='upper right')
sns.despine(fig=fig, right=True, top=True)
fig.tight_layout()
save(fig, 'figS2_loss_terms.svg')
plt.show()

## Reported values

Accuracy is taken at the **final logged epoch** over all runs.
The logistic fits use the **entire trajectory** and exclude any run that
never learned, so their *n* can be smaller.

The derived analysis epoch printed below is the point at which every class has
reached stable performance. It is the epoch the dynamics, trajectory and topology
notebooks evaluate the trained networks at.

In [ ]:
header = f'{"class":<14}{"final accuracy (%)":>24}{"epochs to criterion":>26}'
print(header + '\n' + '-' * len(header))

for d, f in zip(data, fits):
    acc, t = d['final_accuracy'], f['t_conv']
    print(f'{d["label"]:<14}'
          f'{acc.mean():>11.2f} ± {acc.std(ddof=1):<5.2f} (n={acc.size:>3})'
          f'{t.mean():>12,.0f} ± {t.std(ddof=1):<6,.0f} (n={t.size:>3})')

# Accuracy is reported over every run; the fits additionally drop runs that
# never learned (accuracy at or below chance), hence the differing n.
for d, f in zip(data, fits):
    if f['n_excluded']:
        dropped = ', '.join(f'run {i} ({d["final_accuracy"][i]:.0f}%)'
                            for i in f['excluded'])
        print(f'\n{d["label"]}: {f["n_excluded"]} run(s) excluded from the fits '
              f'for never learning — {dropped}.')

print(f'\nDerived analysis epoch for the other notebooks: '
      f'{perf.derive_analysis_epoch(criteria):,}')